In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from langchain.tools import tool
from pydantic import BaseModel, Field

# Works when the notebook is run from the project root
# or directly from src/.
cwd = Path.cwd()
project_root = cwd if (cwd / "src").is_dir() else cwd.parent

sys.path.insert(0, str(project_root / "src"))

load_dotenv(project_root / ".env")

from github_client import GitHubClient

github = GitHubClient()

OWNER = os.environ["GITHUB_OWNER"]
REPO = os.environ["GITHUB_REPO"]
PR_NUMBER = int(os.environ["GITHUB_PR_NUMBER"])

In [ ]:
from typing import Literal


class ListPRInput(BaseModel):
    owner: str = Field(description="GitHub repository owner")
    repo: str = Field(description="GitHub repository name")

    state: Literal["open", "closed", "all"] = "open"

    head: str | None = Field(
        default=None,
        description=(
            "Source branch. For forks use owner:branch."
        )
    )

    base: str | None = Field(
        default=None,
        description="Target branch"
    )


class PRInput(BaseModel):
    owner: str = Field(description="GitHub repository owner")
    repo: str = Field(description="GitHub repository name")

    pr_number: int = Field(
        gt=0,
        description="Pull request number"
    )

In [ ]:
@tool(
    "list_pull_requests",
    args_schema=ListPRInput
)
def list_prs_tool(
    owner: str,
    repo: str,
    state: str = "open",
    head: str | None = None,
    base: str | None = None
) -> list[dict]:
    """
    Find pull requests in a GitHub repository.

    Filter by source branch, target branch or PR state.
    Use this when the user has not provided a PR number.
    """
    return github.list_pull_requests(
        owner=owner,
        repo=repo,
        state=state,
        head=head,
        base=base
    )


@tool(
    "get_pull_request",
    args_schema=PRInput
)
def get_pr_tool(
    owner: str,
    repo: str,
    pr_number: int
) -> dict:
    """
    Retrieve PR metadata, branches and commit SHAs.

    Use this after identifying the PR number.
    """
    return github.get_pull_request(
        owner=owner,
        repo=repo,
        pr_number=pr_number
    )


@tool(
    "get_pr_changed_files",
    args_schema=PRInput
)
def get_changed_files_tool(
    owner: str,
    repo: str,
    pr_number: int
) -> list[dict]:
    """
    Retrieve changed files and available code patches.

    This is the primary source of code changes for review.
    """
    return github.get_pr_changed_files(
        owner=owner,
        repo=repo,
        pr_number=pr_number
    )


@tool(
    "get_pr_diff",
    args_schema=PRInput
)
def get_diff_tool(
    owner: str,
    repo: str,
    pr_number: int
) -> str:
    """
    Retrieve the complete unified diff of a GitHub PR.

    Use when complete diff context is required.
    """
    return github.get_pr_diff(
        owner=owner,
        repo=repo,
        pr_number=pr_number
    )

In [ ]:
github_tools = [
    list_prs_tool,
    get_pr_tool,
    get_changed_files_tool,
    get_diff_tool
]

for github_tool in github_tools:
    print(github_tool.name)
    print(github_tool.description)
    print()

In [ ]:
prs = list_prs_tool.invoke({
    "owner": OWNER,
    "repo": REPO,
    "state": "open"
})

prs

In [ ]:
pr_args = {
    "owner": OWNER,
    "repo": REPO,
    "pr_number": PR_NUMBER
}

pr = get_pr_tool.invoke(pr_args)

print("Title:", pr["title"])
print("Head SHA:", pr["head_sha"])
print("Base SHA:", pr["base_sha"])

In [ ]:
changed_files = get_changed_files_tool.invoke(pr_args)

for file in changed_files:
    print(file["filename"])
    print(file["status"])
    print(file["patch"])
    print("-" * 60)

In [ ]:
diff = get_diff_tool.invoke(pr_args)

print(diff[:5000])

In [10]:
import json

for github_tool in github_tools:
    print(f"\nTool: {github_tool.name}")

    print(
        json.dumps(
            github_tool.args,
            indent=2
        )
    )


Tool: list_pull_requests
{
  "owner": {
    "description": "GitHub repository owner",
    "title": "Owner",
    "type": "string"
  },
  "repo": {
    "description": "GitHub repository name",
    "title": "Repo",
    "type": "string"
  },
  "state": {
    "default": "open",
    "enum": [
      "open",
      "closed",
      "all"
    ],
    "title": "State",
    "type": "string"
  },
  "head": {
    "anyOf": [
      {
        "type": "string"
      },
      {
        "type": "null"
      }
    ],
    "default": null,
    "description": "Source branch. For forks use owner:branch.",
    "title": "Head"
  },
  "base": {
    "anyOf": [
      {
        "type": "string"
      },
      {
        "type": "null"
      }
    ],
    "default": null,
    "description": "Target branch",
    "title": "Base"
  }
}

Tool: get_pull_request
{
  "owner": {
    "description": "GitHub repository owner",
    "title": "Owner",
    "type": "string"
  },
  "repo": {
    "description": "GitHub repository name",
